In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import pickle

df = pd.read_pickle('../data/processed/emails_preprocessed.pkl')
print(f"Loaded {len(df):,} emails")

Loaded 458,386 emails


In [2]:
# TF-IDF matrix
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2)
)
tfidf_matrix = tfidf.fit_transform(df['processed_text'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# Save for later use
scipy.sparse.save_npz('../data/features/tfidf_matrix.npz', tfidf_matrix)
with open('../data/features/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("TF-IDF saved successfully")

TF-IDF matrix shape: (458386, 5000)
TF-IDF saved successfully


In [3]:
def extract_stylistic_features(text):
    """Extract writing style features for anomaly detection."""
    if not isinstance(text, str) or len(text) == 0:
        return {}
    
    words = text.split()
    sentences = text.split('.')
    
    return {
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'avg_sentence_length': np.mean([len(s.split()) for s in sentences if s.strip()]) if sentences else 0,
        'vocab_richness': len(set(words)) / len(words) if words else 0,
        'num_words': len(words),
        'num_sentences': len([s for s in sentences if s.strip()]),
        'exclamation_ratio': text.count('!') / len(text),
        'question_ratio': text.count('?') / len(text),
        'uppercase_ratio': sum(1 for c in text if c.isupper()) / len(text),
        'digit_ratio': sum(1 for c in text if c.isdigit()) / len(text),
    }

from tqdm import tqdm
tqdm.pandas()

style_features = df['clean_body'].progress_apply(extract_stylistic_features)
style_df = pd.DataFrame(style_features.tolist())
style_df.index = df.index

# Save
style_df.to_pickle('../data/features/stylistic_features.pkl')
print("Stylistic features saved")
print(style_df.describe())

100%|██████████| 458386/458386 [00:47<00:00, 9566.10it/s] 


Stylistic features saved
       avg_word_length  avg_sentence_length  vocab_richness      num_words  \
count    458386.000000        458386.000000   458386.000000  458386.000000   
mean          4.965252            14.816158        0.806438     190.826251   
std           1.263855            14.174007        0.158713     759.486030   
min           1.325625             1.000000        0.018367       1.000000   
25%           4.396226             8.750000        0.697778      25.000000   
50%           4.829341            13.000000        0.829268      64.000000   
75%           5.298701            18.125000        0.940000     159.000000   
max         112.000000          3657.666667        1.000000   62487.000000   

       num_sentences  exclamation_ratio  question_ratio  uppercase_ratio  \
count  458386.000000      458386.000000   458386.000000    458386.000000   
mean       12.058169           0.001076        0.001938         0.068540   
std        44.260080           0.007919     

In [4]:
from transformers import AutoTokenizer, AutoModel
import torch

# Use a smaller BERT model for efficiency on MacBook
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text, max_length=512):
    """Get BERT sentence embedding."""
    inputs = tokenizer(text[:2000], return_tensors='pt',
                       truncation=True, max_length=max_length,
                       padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

# Smart sample: all key users + random others
key_users = ['lay-k', 'skilling-j', 'fastow-a', 'delainey-d', 'rice-k', 'causey-r', 'kean-s', 'kitchen-l', 'lavorato-j']
key_emails = df[df['user'].isin(key_users)]
other_emails = df[~df['user'].isin(key_users)].sample(min(15000, len(df)), random_state=42)
sample = pd.concat([key_emails, other_emails]).drop_duplicates()

print(f"Sample size: {len(sample):,}")
print(f"Key user emails: {len(key_emails):,}")
print(f"Other emails: {len(other_emails):,}")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample size: 59,164
Key user emails: 44,164
Other emails: 15,000


In [5]:
from tqdm import tqdm

embeddings = []
indices = []
for idx in tqdm(sample.index, desc="Computing BERT embeddings"):
    try:
        emb = get_embedding(df.loc[idx, 'clean_body'])
        embeddings.append(emb)
        indices.append(idx)
    except:
        continue

embeddings_array = np.array(embeddings)
np.save('../data/features/bert_embeddings.npy', embeddings_array)
np.save('../data/features/bert_embedding_indices.npy', np.array(indices))
print(f"Saved {len(embeddings)} BERT embeddings, shape: {embeddings_array.shape}")

Computing BERT embeddings: 100%|██████████| 59164/59164 [51:25<00:00, 19.17it/s]  


Saved 59164 BERT embeddings, shape: (59164, 384)
